# Linear Regression
---

Given data $(x_1, \ldots, x_n)$ with $x_i \in \mathbb{R}^d$ and targets $(y_1, \ldots, y_n)$ with $y_i \in \mathbb{R}^m$, we want a hypothesis $f : \mathcal{X} \to \mathcal{Y}$ that maps an input $x$ to a prediction $f(x)$.

Regression models have continuous outputs $y \in \mathbb{R}^m$. We assume the hypothesis is linear in the parameters:

$$
f(x) = W^\top x, \quad W \in \mathbb{R}^{d \times m},
$$

so $y$ is a linear function of $W$. The inputs may enter through basis functions $\phi(x)$; replacing $x$ with $\phi(x)$ gives a generalized linear model.

---

The model above is deterministic: each $x$ maps to a single $f(x)$. In practice, observations are noisy and are modelled as

$$
y_i = f(x_i) + \epsilon_i, \quad i = 1, \ldots, n \tag{1}
$$

where the errors $\epsilon_i$ are i.i.d. $\mathcal{N}(0, \sigma^2)$ and independent of $x_i$. Equivalently we can say that **conditioned on the independent variable $x_i$**, $y_i \mid x_i \sim \mathcal{N}(f(x_i), \sigma^2)$.

When $f$ is the true generating function, the **error** is $\epsilon_i = y_i - f(x_i)$. After fitting $\hat{f}$, the **residual** is $e_i = y_i - \hat{f}(x_i)$; if $\hat{f} = f$, then $e_i = \epsilon_i$. From the noise model, the true errors satisfy:

1. **Homoscedasticity:** $\mathrm{Var}(\epsilon_i \mid x_i) = \sigma^2$ for all $i$ (constant variance).
2. **Independence:** $\epsilon_i$ and $\epsilon_j$ are independent for $i \neq j$.
3. **Normality:** $\epsilon_i \sim \mathcal{N}(0, \sigma^2)$.

These properties are often checked on residuals $e_i$ after fitting.

A separate assumption concerns the design matrix $X \in \mathbb{R}^{n \times d}$, whose rows are $x_i^\top$:

4. **No multicollinearity:** $\mathrm{rank}(X) = d$ (full column rank), i.e. no feature column is a linear combination of the others. If this were not the case we would have redundant features.

---

Given a dataset $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^n$, stack the inputs into a design matrix $X \in \mathbb{R}^{n \times d}$ (rows $x_i^\top$) and targets into $Y \in \mathbb{R}^{n \times m}$ (rows $y_i^\top$). The linear model in matrix form is

$$
Y = XW,
$$

equivalently $y_i = W^\top x_i$ for each $i$. The residual for observation $i$ is $e_i = y_i - W^\top x_i$. The joint likelihood of the errors is

$$
p(e_1, \ldots, e_n) = \prod_{i=1}^n p(e_i) = \prod_{i=1}^n \mathcal{N}(y_i - W^\top x_i, \sigma^2).
$$

Taking the negative log-likelihood,

$$
-\log p(e_1, \ldots, e_n) = \sum_{i=1}^n \frac{\|y_i - W^\top x_i\|_2^2}{2\sigma^2} + K.
$$

For the maximum likelihood estimate, differentiate the sum of squared errors:

$$
\nabla_W \sum_{i=1}^n \|y_i - W^\top x_i\|_2^2 = \nabla_W \sum_{i=1}^n (y_i - W^\top x_i)^\top (y_i - W^\top x_i) = \nabla_W \sum_{i=1}^n (y_i^\top - x_i^\top W)(y_i - W^\top x_i)
$$

$$
\nabla_W \sum_{i=1}^n \bigl(y_i^\top y_i - y_i^\top W^\top x_i - x_i^\top W y_i + x_i^\top W W^\top x_i\bigr) = \sum_{i=1}^n \bigl(-x_i y_i^\top - x_i y_i^\top + 2 x_i x_i^\top W\bigr)
$$

Setting the gradient to zero and cancelling the factor of 2,

$$
\sum_{i=1}^n x_i y_i^\top = \sum_{i=1}^n x_i x_i^\top W \quad \Rightarrow \quad X^\top Y = X^\top X W \quad \Rightarrow \quad W = (X^\top X)^{-1} X^\top Y.
$$

Here, this estimate of $W$ is called the OLS or Ordinary Least Square estimate. For the term $(X^TX)$ to be invertible No multi-collinearity is a necessary condition.

If $n \times d$ is very large, storing $X$ in memory at once can become costly. We can instead minimize the same objective with gradient descent. The likelihood and normality of the residuals give a loss to minimize: the mean squared error between targets and predictions,

$$
\mathcal{L}(W) = \frac{1}{n}\sum_{i=1}^n \|y_i - W^\top x_i\|_2^2 = \frac{1}{n}\|Y - XW\|_F^2.
$$

**Batch gradient descent** updates all of $W$ using the full dataset:

$$
W^{t+1} = W^t - \eta \, \nabla_W \mathcal{L},
\qquad
\nabla_W \mathcal{L} = \frac{2}{n} X^\top (XW - Y).
$$

**Stochastic gradient descent (SGD)** uses one sample $(x_i, y_i)$ per step with $\nabla_W \|y_i - W^\top x_i\|^2 = 2 x_i (W^\top x_i - y_i)^\top$ (noisier but cheaper per iteration).

**Learning rate $\eta$.** The step size controls how far we move along the negative gradient. If $\eta$ is too large, updates overshoot and the loss can oscillate or diverge; if $\eta$ is too small, convergence is slow. For this quadratic loss, stability roughly requires $\eta < 2 / \lambda_{\max}$, where $\lambda_{\max}$ is the largest eigenvalue of the Hessian $\nabla_W^2 \mathcal{L} = \frac{2}{n} X^\top X$. In practice $\eta$ is tuned (fixed schedule, decay, or line search).

**Feature scaling.** Gradient descent is sensitive to the scale of the columns of $X$: a feature with large magnitude contributes more to $X^\top X$ and yields a steeper direction in $W$, so one coordinate of $W$ can update much faster than another. Standardizing or normalizing features (zero mean, unit variance) often makes optimization more stable. The closed-form OLS solution $\hat{W} = (X^\top X)^{-1} X^\top Y$ **reweights** features through $(X^\top X)^{-1}$, so coefficient estimates change under rescaling—but **predictions** $\hat{y} = X \hat{W}$ are unchanged if each column $x^{(j)}$ is scaled by $c_j$ and the corresponding weights are scaled by $1/c_j$. SGD does not apply that reweighting automatically; without scaling, it may converge slowly even when OLS would give the same fit in one solve.

**Convexity.** $\mathcal{L}(W)$ is a convex function of $W$ (a sum of convex quadratics in $W$). Its Hessian $\frac{2}{n} X^\top X$ is positive semidefinite; if $X$ has full column rank, $\mathcal{L}$ is strictly convex and has a **unique** minimizer. Batch gradient descent with a suitable $\eta$ therefore converges to the same $\hat{W}$ as OLS (up to numerical error). SGD converges in expectation under standard step-size schedules, but individual runs fluctuate around the minimum because of mini-batch noise.

# Properties of the OLS Estimator
---

For a fixed design matrix $X$, $\hat{W} = (X^\top X)^{-1} X^\top Y$. On a new sample from the same model we get a different $\hat{W}'$; we analyze bias and variance **conditional on $X$**.

Assume $Y = X W_\star + \varepsilon$ with $\mathbb{E}[\varepsilon \mid X] = 0$ and $\mathbb{E}[\varepsilon \varepsilon^\top \mid X] = \sigma^2 I_n$ (scalar noise; matrix case analogous).

### 1. Unbiasedness

$$
\hat{W} = (X^\top X)^{-1} X^\top (X W_\star + \varepsilon) = W_\star + (X^\top X)^{-1} X^\top \varepsilon.
$$

Therefore

$$
\mathbb{E}[\hat{W} \mid X] = W_\star.
$$

### 2. Variance (covariance)

From unbiasedness, $\hat{W} - W_\star = (X^\top X)^{-1} X^\top \varepsilon$. For a vector/matrix estimator, use the **covariance** (not $\mathbb{E}[\hat{W}^2] - \mathbb{E}[\hat{W}]^2$ elementwise):

$$
\mathrm{Cov}(\hat{W} \mid X) = \mathbb{E}\bigl[(\hat{W} - W_\star)(\hat{W} - W_\star)^\top \mid X\bigr].
$$

**Scalar case ($m = 1$).** Write $\hat{w} = w_\star + (X^\top X)^{-1} X^\top \varepsilon$. Then

$$
\mathrm{Cov}(\hat{w} \mid X) = (X^\top X)^{-1} X^\top \, \mathbb{E}[\varepsilon \varepsilon^\top \mid X] \, X (X^\top X)^{-1}.
$$

With $\mathbb{E}[\varepsilon \mid X] = 0$ and $\mathbb{E}[\varepsilon \varepsilon^\top \mid X] = \sigma^2 I_n$,

$$
\boxed{\mathrm{Cov}(\hat{w} \mid X) = \sigma^2 (X^\top X)^{-1}}.
$$

*Finishing your expansion.* Substitute $Y = X w_\star + \varepsilon$ into $\hat{w} = (X^\top X)^{-1} X^\top Y$, so only $\varepsilon$ remains in $\hat{w} - w_\star$. Cross-terms vanish when taking $\mathbb{E}[\cdot \mid X]$ because $\mathbb{E}[\varepsilon \mid X] = 0$. The surviving term is

$$
(X^\top X)^{-1} X^\top \varepsilon \varepsilon^\top X (X^\top X)^{-1},
$$

which gives $\sigma^2 (X^\top X)^{-1}$ after expectation. Note $(X^\top X)^{-1}$ appears **once** here; an extra $(X^\top X)^{-1}$ in the middle of the expansion is a common slip.

**Matrix case ($m > 1$).** Each column satisfies $\mathrm{Cov}(\hat{w}^{(j)} \mid X) = \sigma^2 (X^\top X)^{-1}$.

### 3. Gauss–Markov (BLUE)

Among all **linear unbiased** estimators of $w_\star$, OLS has minimum variance (smallest covariance in the Loewner order). The matrix case $\hat{W}$ is analogous.

**Step 1 — Characterize linear unbiased estimators.** Any linear estimator has the form $\tilde{w} = C y$ with $C \in \mathbb{R}^{d \times n}$. Under $y = X w_\star + \varepsilon$ and $\mathbb{E}[\varepsilon \mid X] = 0$,

$$
\mathbb{E}[\tilde{w} \mid X] = C X w_\star.
$$

Unbiasedness ($\mathbb{E}[\tilde{w} \mid X] = w_\star$ for all $w_\star$) requires

$$
C X = I_d.
$$

**Step 2 — Every such estimator is OLS plus extra noise.** Write $C = (X^\top X)^{-1} X^\top + D$. Then $C X = I_d + D X$, so unbiasedness forces $D X = 0$. Substituting into $\tilde{w} = C y$:

$$
\tilde{w} = (X^\top X)^{-1} X^\top (X w_\star + \varepsilon) + D (X w_\star + \varepsilon) = w_\star + (X^\top X)^{-1} X^\top \varepsilon + D \varepsilon.
$$

The first noise term is the OLS fluctuation; $D \varepsilon$ is an extra contribution (and $D X w_\star = 0$ removes $w_\star$ from the $D$ term).

**Step 3 — Compare covariances.** With $\mathrm{Cov}(\varepsilon \mid X) = \sigma^2 I_n$ and $D X = 0$,

$$
\mathrm{Cov}(\tilde{w} \mid X) = \sigma^2 \bigl[(X^\top X)^{-1} X^\top + D\bigr]\bigl[(X^\top X)^{-1} X^\top + D\bigr]^\top = \sigma^2 \bigl[(X^\top X)^{-1} + D D^\top\bigr].
$$

OLS gives $\mathrm{Cov}(\hat{w} \mid X) = \sigma^2 (X^\top X)^{-1}$. Hence

$$
\mathrm{Cov}(\tilde{w} \mid X) - \mathrm{Cov}(\hat{w} \mid X) = \sigma^2 D D^\top \succeq 0.
$$

So $\mathrm{Var}(a^\top \tilde{w} \mid X) \ge \mathrm{Var}(a^\top \hat{w} \mid X)$ for every $a \in \mathbb{R}^d$: OLS is **BLUE** (Best Linear Unbiased Estimator).

> **Scope:** This does not require normality of $\varepsilon$. It does not compare to nonlinear or biased estimators (e.g. ridge can have lower MSE by trading bias for variance).

### 4. Estimating $\sigma^2$

With residuals $e = Y - X \hat{W}$, an unbiased estimator is $\hat{\sigma}^2 = \frac{1}{n - d} \|e\|_F^2$ when $\mathrm{rank}(X) = d$.

# Appendix
---

### A. Why the step size condition is $0 < \eta < \frac{2}{\lambda_{\max}}$

For

$$
\mathcal{L}(W)=\frac{1}{n}\|Y-XW\|_F^2,
$$

the gradient is

$$
\nabla_W\mathcal{L}=\frac{2}{n}X^\top(XW-Y).
$$

Let $W_\star$ be the minimizer (OLS solution), so $X^\top(XW_\star-Y)=0$. Define the error matrix $E_t=W_t-W_\star$. Under batch gradient descent,

$$
W_{t+1}=W_t-\eta\frac{2}{n}X^\top(XW_t-Y)
$$

$$
W_{t+1}-W_\star=W_t-W_\star-\eta\frac{2}{n}X^\top(XW_t-Y).
$$

Add and subtract $XW_\star$ inside the gradient term. Since $X^\top(XW_\star-Y)=0$ at the minimizer,

$$
X^\top(XW_t-Y)=X^\top X(W_t-W_\star)=X^\top X E_t.
$$

Therefore

$$
E_{t+1}=\Bigl(I-\eta H\Bigr)E_t,
\qquad H:=\frac{2}{n}X^\top X.
$$

The object that governs convergence is the **iteration matrix** $M := I - \eta H$, not $H$ alone. If $H u_j = \lambda_j u_j$, then

$$
M u_j = (I - \eta H) u_j = (1 - \eta \lambda_j) u_j,
$$

so $u_j$ is also an eigenvector of $M$ with eigenvalue $\mu_j = 1 - \eta \lambda_j$. (More generally, any matrix polynomial in $H$ shares the same eigenvectors as $H$; here $M$ is the affine map $H \mapsto I - \eta H$.)

> **Note:** $X^\top X$ is positive semidefinite for any $X$. For any $v \in \mathbb{R}^d$, let $t = Xv$. Then
> $$
> v^\top X^\top X v = (Xv)^\top (Xv) = t^\top t = \|Xv\|_2^2 \geq 0.
> $$

Since $H$ is symmetric positive semidefinite, $0 \le \lambda_1 \le \cdots \le \lambda_{\max}$. Decompose the error in the eigenbasis of $H$ (equivalently of $M$):

$$
E_t = \sum_j a_t^{(j)} u_j.
$$

Along eigen-direction $u_j$, one GD step multiplies the coefficient by $\mu_j$:

$$
a_{t+1}^{(j)} = \mu_j a_t^{(j)} = (1 - \eta \lambda_j) a_t^{(j)}.
$$

For this mode to contract, we need $|1-\eta\lambda_j|<1$, i.e.

$$
0<\eta<\frac{2}{\lambda_j}.
$$

To contract all modes simultaneously, enforce the strictest condition:

$$
\boxed{0<\eta<\frac{2}{\lambda_{\max}}}.
$$

At $\eta=2/\lambda_{\max}$ the largest-curvature mode oscillates, and for larger $\eta$ it diverges.

### B. Convexity discussion

The MSE objective is quadratic in $W$:

$$
\mathcal{L}(W)=\frac{1}{n}\|Y-XW\|_F^2.
$$

Its Hessian is constant:

$$
\nabla_W^2\mathcal{L}=\frac{2}{n}X^\top X=H\succeq0,
$$

so $\mathcal{L}$ is convex. Consequences:

- Every local minimum is a global minimum.
- If $\mathrm{rank}(X)=d$, then $X^\top X\succ0$, so $\mathcal{L}$ is **strictly convex** and the minimizer is unique.
- If $\mathrm{rank}(X)<d$, the problem is still convex but not strictly convex: there are infinitely many minimizers (same predictions, different coefficients).

Therefore, with full column rank and a stable step size, batch gradient descent converges to the same unique solution as OLS.